In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Load the graph
import networkx as nx
G = nx.read_gml('../data/graph/LEMD_EGLL_2023_04_01.gml')
# Load node ID mappings
node_list = list(G.nodes())
node_to_idx = {node: idx for idx, node in enumerate(node_list)}
idx_to_node = {idx: node for node, idx in node_to_idx.items()}

In [3]:
from equinox.wind.wind_free import WindFree
wind_model = WindFree()

In [4]:
from equinox.vnav.vnav_performance import Performance 
from equinox.vnav.vnav_profiles_rev1 import NARROW_BODY_JET_CLIMB_PROFILE, NARROW_BODY_JET_CLIMB_VS_PROFILE, NARROW_BODY_JET_DESCENT_PROFILE, NARROW_BODY_JET_DESCENT_VS_PROFILE

# Create a performance model
performance = Performance(
    NARROW_BODY_JET_CLIMB_PROFILE,
    NARROW_BODY_JET_DESCENT_PROFILE,
    NARROW_BODY_JET_CLIMB_VS_PROFILE,
    NARROW_BODY_JET_DESCENT_VS_PROFILE,
    cruise_altitude_ft=35000,
    cruise_speed_kts=450,
)

In [5]:
from equinox.vnav.vnav_performance import get_eta_and_distance_climb, get_eta_and_distance_descent
# Get the performance table for the descent phase
descent_performance_table = get_eta_and_distance_descent(performance, 1000) # altitude (ft), eta (s), along_track_distance (nm)
climb_performance_table = get_eta_and_distance_climb(performance, 1000) # altitude (ft), eta (s), along_track_distance (nm)


In [17]:
from equinox.helpers.datetimeh import datestr_to_seconds_since_midnight
landing_time_str = "2024-04-01 12:00:00"
landing_seconds_since_midnight = datestr_to_seconds_since_midnight(landing_time_str)
provisioned_takeoff_time_str = "2024-04-01 10:05:00"
takeoff_time_seconds_since_midnight = datestr_to_seconds_since_midnight(provisioned_takeoff_time_str)
print(f'Target landing time: {landing_time_str} which is {landing_seconds_since_midnight} seconds since midnight.')
print(f'Provisioned take off time: {provisioned_takeoff_time_str} which is {takeoff_time_seconds_since_midnight} seconds since midnight.')

Target landing time: 2024-04-01 12:00:00 which is 43200.0 seconds since midnight.
Provisioned take off time: 2024-04-01 10:05:00 which is 36300.0 seconds since midnight.


# Backward Dynamic Programming with Cost and Gradient Tensors

In [18]:
# Load the cost components
from equinox.cost.cost_model_1 import cost_model_1
cost_model_instance = cost_model_1


In [19]:
import importlib
import equinox.dp.backward_dp_w_grad
import numpy as np
importlib.reload(equinox.dp.backward_dp_w_grad)
run_backward_dp = equinox.dp.backward_dp_w_grad.run_backward_dp



# Load the distance matrix
dist_matrix = np.load("../data/graph/LEMD_EGLL_2023_04_01_distances.npy")

# Load the airspace charges matrix
ac_matrix = np.load("../data/graph/LEMD_EGLL_2023_04_01_charges.npy")

device = "cpu"

V_final, eta_final, alt_final, phase_final, edge_costs_time_binned, edge_cost_gradients_time_binned, edge_to_canonical_idx = run_backward_dp(
            graph=G,
            goal_node_id="EGLL", # Destination for the flight
            estimated_landing_time_str=landing_time_str,
            origin_elevation_ft=0.0, # Assuming LEMD at sea level for profile context
            destination_elevation_ft=0.0, # Assuming EGLL at sea level
            cost_model=cost_model_instance,
            wind_model=wind_model,
            performance_model=performance,
            dist_matrix_np=dist_matrix,
            ac_matrix_np=ac_matrix,
            final_alt_ft=0.0, # Altitude at EGLL at landing time, defaults to destination_elevation_ft if 0.0
            delta_t_seconds=600, # time window length
            max_flight_duration_hours=5, # max duration to consider for time bins
            device=device,
            temperature=5e-3,
            takeoff_eta=takeoff_time_seconds_since_midnight
        )

Topological Generations (Backward): 100%|██████████| 145/145 [00:02<00:00, 57.96it/s]


Some shape information:
- `edge_costs_time_binned: (num_edges, num_time_bins)`
- `edge_cost_gradients_time_binned: (num_edges, num_time_bins, num_params)`

# Inspection of the final value function

In [27]:
V_final[node_to_idx['INCEF']]

tensor([    inf,     inf,     inf,     inf,     inf,     inf,     inf,     inf,
            inf,     inf,     inf,     inf,     inf,     inf,     inf,     inf,
            inf,     inf, -0.2025, -0.1616, -0.1560, -0.1543, -0.1283,     inf,
            inf,     inf,     inf,     inf,     inf,     inf,     inf],
       dtype=torch.float64)